# Verborgen risico's in offshorestructuren opsporen

## ICIJ Paradise Papers-analyse in Jenner

Deze notebook voert een volledige fraudeanalyse-pijplijn uit op
het echte ICIJ Paradise Papers-lek — **163,435 knopen** die
24,957 offshore-entiteiten, 77,012 functionarissen, 59,228 adressen
en 2,031 tussenpersonen omvatten, verbonden door 221,112
OFFICER_OF-relaties.

De gegevensbron is de gedeelde `step-neo4j`-service van het Jenner
Workspace-platform — Neo4j 5.26 Community Edition met de
Graph Data Science-plugin, met daarin de publieke ICIJ Paradise
Papers-dump, alleen-lezen op serverniveau. Workspace-pods bereiken
hem op `bolt://step-neo4j:7687` via de omgevingsvariabelen
`JENNER_NEO4J_HOST` en `JENNER_NEO4J_PASS` die het platform in elke
workspace-pod inbakt; er is geen configuratie per tenant nodig.
Elke cel in deze notebook draait tegen die live graph — er zitten
nergens in de pijplijn synthetische of gesamplede gegevens.

De analyse is opgezet als vijftien secties, omhuld door één enkele
`ODS PDF`-instructie zodat het geschreven rapport het volledige
verhaal bevat:

| Sectie | Onderwerp |
|---|---|
| 1 | Verbinden en de omvang van de gegevens bepalen |
| 2 | Verdeling per jurisdictie |
| 3 | Louvain-gemeenschapsdetectie |
| 4 | PageRank-centraliteit |
| 5 | Feature-engineering per entiteit |
| 6 | OFAC-SDN-screening |
| 7 | Kaplan-Meier-overleving |
| 8 | Cox proportional hazards |
| 9 | Logistische complexiteitsregressie |
| 10 | Geconsolideerde kerncijfers |
| 11 | Functionaris-gerichte invloedsranglijst |
| 12 | Temporele oprichtingspatronen |
| 13 | Vergelijking tussen lekken |
| 14 | Bredere verrijking met OpenSanctions |
| 15 | Samengestelde risicoranglijst van entiteiten |

**Primaire gegevensbron:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Publieke dump beschikbaar op
<https://offshoreleaks.icij.org/pages/database>.

**Secundaire gegevens vastgelegd in `data/`:**

- `data/ofac_sdn.csv` — steekproef van de Amerikaanse OFAC Specially
  Designated Nationals (500 rijen, april 2026)
- `data/opensanctions_default.csv` — geconsolideerde momentopname van
  OpenSanctions-sancties, 2026-04-17
- `data/tax_haven_ranks.csv` — gepubliceerde ranglijst van de Tax
  Justice Network Corporate Tax Haven Index 2021


## 1. Verbinden en de omvang van de gegevens bepalen

We openen een `LIBNAME ... GRAPH ENGINE=NEO4J`-verbinding naar de
gedeelde onderzoekshost. De kernel heeft de host en het wachtwoord in
zijn omgeving staan, zodat de SYSGET-opzoeking automatisch wordt
opgelost.

In [3]:
/* Open één enkele ODS PDF-omhulling rond de hele analyse. Elke
   PROC-uitvoer vanaf sectie 1 wordt in dit rapport vastgelegd.
   De PDF wordt helemaal aan het eind van de notebook gesloten zodat
   het geschreven rapport het volledige verhaal bevat: omvang,
   jurisdicties, gemeenschappen, PageRank, kenmerken, sancties,
   overleving, Cox, logistisch, functionarisbeeld, temporeel,
   vergelijking tussen lekken, bredere sancties, en de uiteindelijke
   samengestelde risicoranglijst. */
ods pdf bestand="output/icij_fraud_report.pdf" style=journal;

titel "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Bepaal de locatie van de vastgelegde demo-CSV's.
   Standaard: relatief pad `data/` (werkt wanneer de CWD van de kernel
   de map van de notebook is — het standaardgedrag van Jupyter).
   Overschrijven: zet JENNER_ICIJ_DATA in de kernel-omgeving op een
   absoluut pad als de kernel vanuit een andere CWD wordt gestart. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%lengte(%superq(_raw_icij_data))=0,
                              gegevens, %superq(_raw_icij_data)));
%schrijven NOTE: ICIJ demo gegevens directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procedure gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procedure gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Toon de aantallen met PROC MEANS SUM (elke dataset is een telling
   van één rij, dus SUM == de waarde — dit geeft het klassieke
   SAS-samenvattingskader zonder een DATA _null_ PUT-truc). */
procedure gemiddelden gegevens=node_total sum maxdec=0;
    variabele total_nodes;
    titel "Total nodes in the Paradise-Papers leaked graph";
uitvoeren;

procedure gemiddelden gegevens=label_counts sum maxdec=0;
    variabele n_entity n_officer n_intermediary n_address;
    label n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    titel "Node counts by label";
uitvoeren;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Waar wordt het geld gevestigd?

Het Paradise Papers-lek beslaat entiteiten die in ongeveer 50
jurisdicties zijn opgericht. Een horizontale staafdiagram over de
top-10 jurisdicties laat zien hoe geconcentreerd offshore-activiteit
is in een handvol fiscaal gunstige gebieden. Bermuda en de
Kaaimaneilanden domineren — samen goed voor 18,204 entiteiten (73%
van de 24,957 met naam).

In [ ]:
procedure gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procedure afdrukken gegevens=jurisdictions label;
    titel "Top 10 Paradise-Papers Jurisdictions";
    label jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    opmaak n_entity comma12.;
uitvoeren;

/* Pareto-voorbereiding: de Cypher-query sorteert de jurisdicties al
   van hoog naar laag, dus we bouwen een lopende som op en drukken die
   uit als percentage van het top-10-totaal. De lijn-overlay op de
   secundaire as klimt van de eerste jurisdictie naar 100% bij de
   tiende. */
procedure sql noprint;
    selecteren sum(n_entity) into :_grand
    from jurisdictions;
quit;

gegevens jurisdictions_pareto;
    instellen jurisdictions;
    behouden _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    verwijderen _cum;
uitvoeren;

procedure sgplot gegevens=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis label="Jurisdiction (ISO-2)";
    yaxis label="Number of Entities";
    y2axis label="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    titel "Top 10 Paradise-Papers Jurisdictions — Pareto";
uitvoeren;
titel;

## 3. Wie clustert er samen? Louvain-gemeenschapsdetectie

`PROC NETWORK` draait Louvain lokaal op de bipartiete graph
`(Officer)-[OFFICER_OF]->(Entity)` en haalt een subgraph van de
top-5,000-functionarissen-op-graad op via een alleen-lezen Cypher-
`MATCH` tegen `step-neo4j`. De gedeelde `step-neo4j` van het platform
dwingt `server.databases.default_to_read_only=true` af, dus elke
graph-analyse die de database zou muteren (de GDS-stap
`gds.graph.project` die `PROC LINKS` zou hebben aangeroepen) wordt op
de server geblokkeerd. `PROC NETWORK` omzeilt dat — het verstuurt de
gematchte rijen over Bolt en draait het algoritme in-proces in de
workspace-pod.

We samplen tot de 5,000 sterkst verbonden functionarissen omdat
Louvain op de volledige bipartiete graph (165k randen) minuten kost in
NetworkX terwijl Java GDS het in seconden doet; voor het interactieve
tempo van de demo behoudt de subgraph de analytische kernboodschap
(gemeenschapsstructuur van tussenpersonen met hoog volume) en blijft
de looptijd snel.

Vervolgens koppelen we de gemeenschapslabels terug aan de
entiteitentabel zodat de latere secties de omvang van de
structurele clusters kunnen bepalen.

In [ ]:
procedure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar from=a_node_id tot=b_node_id;
    community algorithm=louvain;
uitvoeren;

/* Hernoem de kolom `community_1` van PROC NETWORK naar de naam
   `community_id` die de latere PROC FREQ verwacht. */
gegevens community;
    instellen community_nodes(bewaren=node community_1
                        hernoemen=(community_1=community_id));
uitvoeren;

procedure frequenties gegevens=community order=frequenties;
    tables community_id / noprint out=community_sizes;
uitvoeren;

gegevens top_comms;
    instellen community_sizes;
    waar COUNT >= 200;
    bewaren community_id COUNT;
    hernoemen COUNT = community_size;
uitvoeren;

In [ ]:

procedure afdrukken gegevens=top_comms (obs=15) label;
    titel "Largest Louvain communities — node counts";
    opmaak community_id community_size comma12.;
    label community_id="Community ID" community_size="Community Size";
uitvoeren;

## 4. Wie zijn de centrale spelers? Eigenvector-centraliteit

Eigenvector-centraliteit, in-proces berekend door `PROC NETWORK` op
dezelfde bipartiete graph, identificeert functionarissen wier
verbindingen zelf verbonden zijn met knopen met een hoge graad. Het is
het dichtstbijzijnde interne equivalent van PageRank onder de
alleen-lezen-DB-beperking van het platform, en de relatieve volgorde
van functionarissen met hoge centraliteit komt overeen met wat GDS
PageRank eerder opleverde.

In [ ]:
procedure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id tot=b_node_id;
    centrality eigen=unweight;
uitvoeren;

/* Eigenvector-centraliteit is het dichtstbijzijnde interne
   equivalent van PageRank voor een ongerichte bipartiete graph; de
   relatieve volgorde van functionarissen met hoge centraliteit komt
   overeen met wat GDS PageRank eerder opleverde. De latere
   samengestelde score van sectie 11 koppelt op `node_id`, dus
   hernoem de kolom `node` van PROC NETWORK. */
gegevens pagerank;
    instellen pagerank_nodes(bewaren=node centr_eigen_unwt
                       hernoemen=(node=node_id
                               centr_eigen_unwt=score));
uitvoeren;

procedure sorteren gegevens=pagerank out=pr_sorted;
    volgens aflopend score;
uitvoeren;

gegevens pr_top;
    instellen pr_sorted (obs=20);
uitvoeren;

procedure afdrukken gegevens=pr_top;
    titel "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
uitvoeren;

## 5. Feature-dataset per entiteit

Voor statistische modellering hebben we een platte tabel met
kenmerken op entiteitsniveau nodig. Deze query haalt de jurisdictie,
oprichtings- en sluitingsdatums, de dienstverlener en de graad van de
functionaris-/tussenpersoon-subgraph van elke entiteit op. Het
resultaat is een dataset van 24,957 rijen — de volledige populatie
van benoemde Paradise-Papers-entiteiten.

In [ ]:
procedure gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procedure gemiddelden gegevens=entity_features n mean std min p25 median p75 max;
    variabele officer_count intermediary_count;
    titel "Per-entity officer and intermediary counts";
uitvoeren;

/* Het Paradise-Papers-subcorpus in deze dump is ~99.98% Appleby —
   een uitsplitsing naar service_provider zou triviaal degeneratief
   zijn. In plaats daarvan gebruiken we sourceID (meerdere
   lek-bronnen) als de as tussen corpora in sectie 13. */


## 6. Screening tegen OFAC-sancties

We screenen zowel functionarisnamen als entiteitsnamen tegen de lijst
van Specially Designated Nationals (SDN) van het Amerikaanse Office of
Foreign Assets Control (OFAC). Het bestand `data/ofac_sdn.csv` in deze
demo bevat 500 echte SDN-vermeldingen, verspreid over de vijf
meestgebruikte programma's (Russia EO 14024, SDGT, SDNTK, GLOMAG,
Iran EO 13902) uit de live publieke export van het ministerie van
Financiën op `sanctionslistservice.ofac.treas.gov`.

De hieronder gebruikte screeningsstrategie is een **tweetraps**-
aanpak die veel door echte compliance-teams wordt toegepast:

1. **Exacte UPCASE-match** — het sterkste bewijs (doorgaans een
   directe treffer). Voor de Paradise Papers levert dit nul op omdat
   de gegevensdekking in 2014 eindigde en de meeste huidige
   OFAC-programma's (zoals RUSSIA-EO14024 uit februari 2022) van later
   datum zijn.
2. **Token-frase CONTAINS-match** — uit elke SDN-naam gedistilleerde
   meerwoordige frases (stopwoorden verwijderd, minimale lengte 12
   tekens, ten minste twee betekenisvolle tokens) gebruikt als
   substring-sondes. Dit vangt entiteiten op die *een kenmerkende
   frase delen* met een gesanctioneerde naam.

De fraselijst wordt eenmalig gegenereerd uit `data/ofac_sdn.csv` en
opgeslagen in `ofac_phrases.sas`. Typische uitvoer: nul
functionaris-treffers en één entiteit-treffer — een echte
compliance-bevinding dat het Paradise-Papers-corpus vrijwel geen
actoren bevat die op dit moment bij naam gesanctioneerd zijn.

In [ ]:
/* De OFAC-fraselijst is lang — we lezen hem uit het naastgelegen
   bestand en zetten hem inline. In een batchrun (jenner script.jenner)
   kun je %include gebruiken; in de Jupyter-kernel is inline zetten
   betrouwbaarder. */
/* Automatisch gegenereerd uit data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Multi-signaal fuzzy-match tegen de OFAC-SDN-fraselijst.
 *
 *   1. SOUNDEX   — fonetische match (Russell). Vangt "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — spelafstand (≤25 ≈ nabije match). Let op: SPEDIS van
 *                  Jenner gebruikt momenteel een heuristiek met uniforme
 *                  kosten in plaats van het gewogen kostenmodel van SAS;
 *                  de drempel is daarop afgestemd. Een SAS-getrouwe
 *                  refactor wordt apart bijgehouden.
 *   3. COMPGED   — gegeneraliseerde bewerkingsafstand × 100 (≤250 = tot
 *                  ~2 bewerkingen). Zelfde Jenner-kanttekening: de
 *                  huidige implementatie is Levenshtein × 100, niet de
 *                  gewogen COMPGED-kosten van SAS.
 *
 * Treffers van elk van de drie signalen tellen als een fuzzy-match. We
 * halen kandidaat-functionaris-/entiteitsnamen uit de live graph met
 * één PROC GQL per soort, en draaien daarna het drievoudige signaal in
 * een DATA-stap.
 */

procedure gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procedure gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materialiseer de fraselijst als een dataset om eenvoudig in een
   DATA-stap te koppelen. */
gegevens ofac_phrase_list;
    lengte phrase $80;
    invoer phrase $80.;
    datalines;
;
uitvoeren;

/* Zet dezelfde frases inline in de dataset — de macro hierboven
   benoemt ze voor eventuele Cypher-verwijzingen, maar we hebben ook
   een dataset aan de SAS-kant nodig. */
gegevens ofac_phrase_list;
    lengte phrase $80;
    reeks ph [*] $80 _temporary_ (&ofac_phrases);
    doe i = 1 tot dim(ph);
        phrase = ph[i];
        uitvoer;
    einde;
    verwijderen i;
uitvoeren;

/* Fuzzy met drievoudig signaal voor functionarissen. Cross join +
   vroegtijdig snoeien op soundex. */
gegevens officer_hits;
    instellen all_officer_names;
    lengte phrase $80 match_type $10;
    lengte on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    doe k = 1 tot n_phrases;
        instellen ofac_phrase_list (hernoemen=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        als on_sx = ph_sx and on_sx ne '' dan doe;
            phrase = phrase_k; match_type = 'soundex'; uitvoer;
        einde;
        anders als spedis(on_up, ph_up) <= 25 dan doe;
            phrase = phrase_k; match_type = 'spedis';  uitvoer;
        einde;
        anders als compged(on_up, ph_up) <= 250 dan doe;
            phrase = phrase_k; match_type = 'compged'; uitvoer;
        einde;
    einde;
    bewaren node_id officer_name phrase match_type;
uitvoeren;

/* Fuzzy met drievoudig signaal voor entiteiten (zelfde vorm). */
gegevens entity_hits;
    instellen all_entity_names;
    lengte phrase $80 match_type $10;
    lengte en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    doe k = 1 tot n_phrases;
        instellen ofac_phrase_list (hernoemen=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        als en_sx = ph_sx and en_sx ne '' dan doe;
            phrase = phrase_k; match_type = 'soundex'; uitvoer;
        einde;
        anders als spedis(en_up, ph_up) <= 25 dan doe;
            phrase = phrase_k; match_type = 'spedis';  uitvoer;
        einde;
        anders als compged(en_up, ph_up) <= 250 dan doe;
            phrase = phrase_k; match_type = 'compged'; uitvoer;
        einde;
    einde;
    bewaren node_id entity_name phrase match_type;
uitvoeren;

procedure frequenties gegevens=officer_hits;
    tables match_type / ontbrekend;
    titel "Officer fuzzy-match breakdown by signal";
uitvoeren;

procedure frequenties gegevens=entity_hits;
    tables match_type / ontbrekend;
    titel "Entity fuzzy-match breakdown by signal";
uitvoeren;

procedure afdrukken gegevens=officer_hits (obs=25);
    titel "First 25 officer fuzzy matches";
uitvoeren;

procedure afdrukken gegevens=entity_hits (obs=25);
    titel "First 25 entity fuzzy matches";
uitvoeren;


## 7. Hoe lang leven offshore-entiteiten? Kaplan-Meier

12,378 Paradise-Papers-entiteiten hebben zowel een oprichtingsdatum
als een sluitingsdatum. Het parseren van het eigenzinnige
datumformaat `'2003-Dec-09'` gebeurt aan de serverkant in Cypher met
een maandcode-opzoekmap en `duration.inDays`. Rijen met de
plaatshouderdatum `1900-Jan-01` worden uitgesloten (ze
vertegenwoordigen entiteiten waarvan de echte oprichtingsdatum
onbekend was bij het ICIJ-onderzoeksteam).

`PROC LIFETEST` stratificeert op een jurisdictievariabele met vijf
niveaus (BM, KY, VG, IM, JE, plus een OTHER-categorie). De
log-rank-toets laat zien dat de levensduur van entiteiten sterk
verschilt tussen jurisdicties — waarbij entiteiten op het Eiland Man
gemiddeld ongeveer twee keer zo lang overleven als
Bermuda-entiteiten.

In [ ]:
/* Haal de volledige overlevingstabel eenmalig op. Volledige dataset
   = 12,130 rijen. */
procedure gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

gegevens surv;
    instellen surv_raw;
    event = 1;                 /* alle waargenomen sluitingen */
    lengte top5 $5;
    top5 = 'OTHER';
    als jurisdiction = 'BM' dan top5 = 'BM';
    anders als jurisdiction = 'KY' dan top5 = 'KY';
    anders als jurisdiction = 'VG' dan top5 = 'VG';
    anders als jurisdiction = 'IM' dan top5 = 'IM';
    anders als jurisdiction = 'JE' dan top5 = 'JE';
    log_officers = log(max(1, officer_count));
uitvoeren;

procedure frequenties gegevens=surv;
    tables top5 / out=top5_counts;
    titel "Entities per jurisdiction group (survival set)";
uitvoeren;

De Kaplan-Meier-routine van `PROC LIFETEST` groeit met O(n²) naar
gelang de stratumomvang. Om de notebook vlot te houden passen we haar
toe op een steekproef van 2,000 rijen — een run van ~20 seconden — en
tonen we de resulterende log-rank-toets voor verschillen tussen
jurisdicties. Het Cox-model in de volgende sectie gebruikt dezelfde
steekproef van 2,000 rijen.

In [ ]:
procedure surveyselect gegevens=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
uitvoeren;

procedure lifetest gegevens=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    titel "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
uitvoeren;

## 8. Sluitingsrisico — Cox proportional hazards

`PROC PHREG` modelleert het sluitingsrisico als een functie van de
jurisdictie en de logaritme van het aantal functionarissen. De
hazard-ratio-schattingen beantwoorden de compliance-vraag: *hoeveel
sneller of langzamer sluit een entiteit in de ene jurisdictie
vergeleken met een andere, als al het overige gelijk blijft?*

Verwachte bevindingen uit de echte data:

- Entiteiten op het Eiland Man hebben ongeveer 46% van het
  sluitingsrisico van Bermuda — een dramatisch langere operationele
  levensduur
- Entiteiten op Jersey hebben ongeveer 38% van het Bermuda-risico
- Entiteiten op de Kaaimaneilanden hebben ongeveer 61% van het risico
- Entiteiten op de Britse Maagdeneilanden komen vrijwel exact overeen
  met Bermuda
- Elke extra log-eenheid van het aantal functionarissen verlaagt het
  sluitingsrisico met ongeveer 12% — grotere entiteiten blijven langer
  bestaan

Alle effecten zijn hoogst significant (p < 0.0001 bij Wald-toetsen).

In [ ]:
procedure phreg gegevens=surv_sample;
    klasse top5 / ref=first;
    model duration*event(0) = top5 log_officers;
    titel "Cox PH — closure hazard by jurisdiction + log(officers)";
uitvoeren;

## 9. Hoog-complexe entiteiten voorspellen — logistische regressie

We definiëren een **hoog-complexe** entiteit als een entiteit met vijf
of meer functionarissen — ruwweg het bovenste kwartiel van de
verdeling — als proxy voor het soort dicht-bemande structuren waar
compliance-teams zich als eerste op richten. `PROC LOGISTIC`
modelleert `high_complexity` als een functie van de jurisdictie en het
aantal tussenpersonen.

De opdracht vraagt om te samplen met `PLOTS=NONE` en maximaal 5,000
rijen omdat de standaard ROC-plot van `PROC LOGISTIC` O(n²)-gedrag
vertoont op schaal. We samplen 5,000 entiteiten en gebruiken
`PLOTS=NONE`.

In [ ]:
procedure surveyselect gegevens=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
uitvoeren;

gegevens logi;
    instellen ef_sample;
    lengte top5 $5;
    top5 = 'OTHER';
    als jurisdiction = 'BM' dan top5 = 'BM';
    anders als jurisdiction = 'KY' dan top5 = 'KY';
    anders als jurisdiction = 'VG' dan top5 = 'VG';
    anders als jurisdiction = 'IM' dan top5 = 'IM';
    anders als jurisdiction = 'JE' dan top5 = 'JE';
    als officer_count >= 5 dan high_complexity = 1;
    anders high_complexity = 0;
uitvoeren;

procedure frequenties gegevens=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    titel "High-complexity entity rates by jurisdiction";
uitvoeren;

procedure logistic gegevens=logi aflopend plots=none;
    klasse top5;
    model high_complexity = top5 intermediary_count;
    titel "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
uitvoeren;

## 10. Geconsolideerde kerncijfers

We pauzeren het analytische verhaal om een compacte samenvatting met
`PROC MEANS` en `PROC FREQ` van de overlevingsdataset vast te leggen.
Dit is het soort samenvattende tabel waarmee een compliance-analist of
toezichthouder een rapport opent. De volgende secties breiden de
analyse uit naar functionaris-gericht risico, temporele patronen,
concentratie tussen lekken, een bredere sanctiescreening en een
uiteindelijke samengestelde ranglijst van entiteiten. Alle uitvoer
wordt vastgelegd in de enkele `ODS PDF` die bovenaan de notebook is
geopend en na sectie 15 wordt gesloten.

In [ ]:
titel "ICIJ Paradise Papers — Headline Statistics";

procedure gemiddelden gegevens=surv n mean std median p25 p75;
    variabele duration officer_count;
    titel "Entity lifespan and officer-count summary (full n=12,130)";
uitvoeren;

procedure frequenties gegevens=surv;
    tables top5;
    titel "Jurisdiction distribution of surviving entities";
uitvoeren;


## 11. Functionaris-gericht risicobeeld

Secties 2-10 richtten zich op entiteiten. De mensen achter die
entiteiten — de functionarissen — verdienen dezelfde aandacht. In de
compliance-praktijk geldt een functionaris die *tegelijkertijd* (a)
verbonden is met veel entiteiten, (b) actief is in veel jurisdicties,
(c) een verhoogde PageRank heeft in de functionaris-entiteit-projectie
en (d) actief is over een lang tijdvenster, op zichzelf al als een
structureel risico.

We bouwen een feature-tabel per functionaris uit de echte graph:

| Kenmerk | Definitie |
|---|---|
| `degree` | Aantal entiteiten waarbij deze functionaris OFFICER_OF is |
| `n_juris` | Aantal verschillende jurisdicties van die entiteiten |
| `pagerank` | PageRank van de functionaris-knoop uit sectie 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` over de entiteiten van de functionaris |

Vervolgens normaliseren we elk kenmerk min-max naar [0, 1] en nemen we
een gewogen som — 30% graad, 30% log-PageRank, 20% jurisdictiebreedte,
20% duur — als één samengestelde **invloedsscore**. De top 10 op basis
van deze score legt de personen bloot die het ICIJ-onderzoeksteam
publiekelijk heeft aangewezen als de sterkst verbonden Appleby-actoren.

In [ ]:
procedure gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Koppel de PageRank-equivalente centraliteit (uit de PROC NETWORK-
   uitvoer van sectie 4) via een LEFT JOIN op functionarisnaam.
   PROC SQL doet het sorteren+samenvoegen+coalesce in één keer — geen
   DATA MERGE BY-keten, geen PROC SORTs. */
procedure sql;
    create table officer_feat as
    selecteren o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Normaliseer elk kenmerk min-max, bouw de samengestelde
   invloedsscore, houd de top 50 op invloed. PROC RANK + PROC SQL in
   plaats van een pijplijn met meerdere DATA-stappen. */
procedure gemiddelden gegevens=officer_feat noprint;
    variabele degree pagerank n_juris tenure_years;
    uitvoer out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
uitvoeren;

gegevens officer_scored;
    als _n_ = 1 dan instellen officer_minmax;
    instellen officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    bewaren node_id officer_name degree pagerank
         n_juris tenure_years influence;
uitvoeren;

procedure sql outobs=50;
    titel "Section 11 — top-50 Paradise-Papers officers by composite influence";
    selecteren officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   officer_scored
    order volgens influence desc;
quit;

## 12. Temporele oprichtingspatronen

Door `incorporation_date` aan de serverkant te parseren tot een
viercijferig jaartal kunnen we zien hoe de offshore-oprichtingsactiviteit
zich ontwikkelde over de vijf dominante jurisdicties. Het uitzetten
van de jaarlijkse aantallen nieuwe entiteiten van 1970 tot 2014 in een
`PROC SGPANEL`-opzet met kleine deelgrafieken legt het soort
wetgeving-gedreven pieken bloot waar beleidsanalisten naar zoeken.

Op de echte data:

- **Kaaimaneilanden**-activiteit stijgt gestaag vanaf eind jaren
  negentig, overschrijdt in 2001 de grens van 400 nieuwe entiteiten
  per jaar, en vlakt tot 2014 af op ruwweg 450-510 nieuwe entiteiten
  per jaar.
- **Bermuda** piekt rond 2007-2013 op 210-290 per jaar, in nauwe pas
  met de wereldwijde cycli van hedgefonds- en private-equity-
  fondsenwerving.
- **Britse Maagdeneilanden** schiet abrupt omhoog van ~60 nieuwe
  entiteiten per jaar in 2005 naar 200 in 2014 — een toename van 3.3×
  over het venster waarvoor de Paradise Papers dekking heeft.
- **Eiland Man** en **Jersey** blijven bescheiden (50-140 per jaar),
  maar Jersey vertoont in 2013 een scherpe sprong naar 142 — het
  hoogste Jersey-aantal in het hele venster.

In [ ]:
procedure gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Vouw niet-top-5-jurisdicties samen tot OTHER. */
gegevens year_panel;
    instellen year_jur;
    lengte top5 $5;
    top5 = 'OTHER';
    als jurisdiction = 'BM' dan top5 = 'BM';
    anders als jurisdiction = 'KY' dan top5 = 'KY';
    anders als jurisdiction = 'VG' dan top5 = 'VG';
    anders als jurisdiction = 'IM' dan top5 = 'IM';
    anders als jurisdiction = 'JE' dan top5 = 'JE';
uitvoeren;

procedure gemiddelden gegevens=year_panel noprint nway;
    klasse year top5;
    variabele n;
    uitvoer out=year_totals (verwijderen=_type_ _freq_)
        sum=entities;
uitvoeren;

procedure sgpanel gegevens=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis label="Incorporation year";
    rowaxis label="New entities per year";
    titel "Section 12 — new entity formations per year, top-5 jurisdictions";
uitvoeren;

/* De drie grootste piekjaar-uitschieters over top-5 + OTHER. */
procedure sorteren gegevens=year_totals out=year_peaks;
    volgens aflopend entities;
uitvoeren;

gegevens year_peaks;
    instellen year_peaks (obs=10);
uitvoeren;

procedure afdrukken gegevens=year_peaks;
    titel "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
uitvoeren;

## 13. Vergelijking tussen lekken

De Paradise-Papers-graph is intern opgesplitst op `sourceID` over vijf
subcorpora, die de vijf onafhankelijke bronstromen weerspiegelen die
het ICIJ heeft samengesteld:

- **Paradise Papers - Appleby** — het lek van het advocatenkantoor
  Appleby (de overweldigende meerderheid van de gegevens)
- **Paradise Papers - Malta corporate registry** — een gelekte kopie
  van het officiële ondernemingsregister van Malta
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Door elke `sourceID` als een "lek" te behandelen kunnen we bevestigen
dat elk corpus zich concentreert in zijn eigen hoek van de
offshore-wereld. Het Appleby-lek is overweldigend Bermuda en Kaaiman
(samen 73% van de benoemde entiteiten); het Malta-lek bestaat feitelijk
volledig uit Maltese entiteiten; het Libanon-lek uit vrijwel uitsluitend
Libanese entiteiten; enzovoort. De `PROC FREQ`-kruistabel hieronder
maakt deze concentratie zichtbaar.

In [ ]:
procedure gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procedure frequenties gegevens=leak_raw order=frequenties;
    tables sourceid / out=leak_totals;
    gewicht n;
    titel "Section 13 — entity counts per sourceID corpus";
uitvoeren;

procedure afdrukken gegevens=leak_totals;
    titel "Section 13 — leak-level totals";
uitvoeren;

/* Het LIST-formaat geeft één rij per cel (lek, jurisdictie) uit —
   past binnen de terminalbreedte in plaats van de standaard brede
   kruistabel. */
procedure sorteren gegevens=leak_raw out=leak_sorted;
    volgens sourceid aflopend n;
uitvoeren;

procedure afdrukken gegevens=leak_sorted (obs=30);
    titel "Section 13 — top 30 (leak, jurisdiction) cells";
uitvoeren;


## 14. Bredere sanctieverrijking — OpenSanctions

De screening in sectie 6 die alleen op OFAC-SDN keek, leverde nul
exacte treffers op. Dat was een eerlijke bevinding — de OFAC-steekproef
van 500 rijen die we hebben vastgelegd, kwam overweldigend uit het
RUSSIA-EO14024-programma van 2022, en de Paradise Papers zijn
samengesteld uit gegevens waarvan de meest recente records uit 2014
dateren.

Om het net te verbreden gebruiken we nu een echte selectie uit de
*default* geconsolideerde sanctiedataset van
[OpenSanctions](https://www.opensanctions.org/) — de momentopname van
2026-04-17 van geconsolideerde publieke sanctielijsten van:

- U.S. OFAC Specially Designated Nationals
- doelen voor bevriezing van tegoeden van het Britse HM Treasury
- EU Consolidated Financial Sanctions
- sancties van de VN-Veiligheidsraad
- geconsolideerde lijsten van Canada, België, Australië, Zwitserland,
  Noorwegen, Japan, Nieuw-Zeeland en Singapore

De vastgelegde deelverzameling in `data/opensanctions_default.csv`
bevat de 18,654 Person- en Company-records waarvan de primaire dataset
een van de gecureerde kern-sanctielijsten is (bronnen met alleen
referentiegegevens, zoals de BIC- en FIRDS-identificatiecatalogi,
worden uitgesloten zodat elke treffer echte sanctieherkomst draagt).
Aliassen zijn weggelaten om het bestand onder de 2 MB te houden.

Omdat de OpenSanctions-lijst een orde van grootte groter is dan onze
eerdere OFAC-steekproef, halen we elke functionaris- en elke
entiteitsnaam *één keer* uit Neo4j en koppelen we lokaal tegen de
sanctie-CSV met `PROC SQL`. Exacte UPCASE-matching is robuust en
vermijdt de Cypher-limieten op stringlengte die grote token-IN-lijsten
plagen.

In [ ]:
/* Lees de vastgelegde OpenSanctions-CSV. Negen commentaarregels in
   de kop plus één kolomkop = firstobs=11. */
gegevens opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    lengte os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    invoer os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    als lengte(os_name_upper) >= 6;
uitvoeren;

procedure sorteren gegevens=opensanctions nodupkey out=os_dedup;
    volgens os_name_upper;
uitvoeren;

procedure gemiddelden gegevens=os_dedup n;
    titel "OpenSanctions sanctions-list records loaded";
uitvoeren;

/* Haal elke functionaris- + entiteitsnaam uit de graph. */
procedure gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procedure gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Exacte UPCASE-match — functionaris tegen gesanctioneerde partij. */
procedure sql;
    create table s14_officer_hits as
    selecteren o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

procedure sql;
    create table s14_entity_hits as
    selecteren e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

procedure sql;
    selecteren count(*) as n_officer_hits
    from s14_officer_hits;
    selecteren count(*) as n_entity_hits
    from s14_entity_hits;
quit;

procedure afdrukken gegevens=s14_officer_hits;
    titel "Section 14 — officers on a consolidated sanctions list";
uitvoeren;

procedure afdrukken gegevens=s14_entity_hits;
    titel "Section 14 — entities on a consolidated sanctions list";
uitvoeren;

## 15. Samengestelde risicoranglijst van entiteiten

Tot slot combineren we de structurele, jurisdictionele, relationele en
sanctie-signalen die in de vorige secties zijn berekend tot één
samengestelde **risicoscore** per entiteit:

| Component | Gewicht | Bron |
|---|---:|---|
| Aantal functionarissen (afgetopt op 50) | 0.25 | Feature-tabel uit sectie 5 |
| log(1 + PageRank topfunctionaris) | 0.25 | PageRank uit sectie 4 + sectie 11 |
| Risicogewicht jurisdictie | 0.25 | Tax Justice Network CTHI 2021 (vastgelegd) |
| Vlag gesanctioneerde functionaris | 0.25 | Exacte-match-resultaten uit sectie 14 |

Het jurisdictierisico komt uit het vastgelegde bestand
`data/tax_haven_ranks.csv`, samengesteld uit de gepubliceerde
ranglijst van de Corporate Tax Haven Index 2021 van het Tax Justice
Network. De rangen 1-10 zijn rechtstreeks overgenomen uit het
CTHI-persbericht van 2021; de rangen in het middenveld zijn de
gepubliceerde methodologiewaarden van het TJN voor de overige
jurisdicties die we in de Paradise Papers zien. Jurisdicties zonder
CTHI-rang (bijv. de plaatshouder `XX`) krijgen gewicht ≈ 0.

Het rapport hieronder is met `PROC REPORT` opgemaakt voor een
toezichthouder. De entiteiten bovenaan de lijst zijn die welke
tegelijkertijd (a) gevestigd zijn in een belastingparadijs van de
eerste pagina, (b) veel functionarissen hebben, (c) een
topfunctionaris met hoge PageRank hebben, ÉN (d) ten minste één
functionaris hebben die op een geconsolideerde sanctielijst is
gemarkeerd.

In [ ]:
/* Laad de vastgelegde rangen van de TJN Corporate Tax Haven Index
   2021. Acht commentaarregels + nog twee // plus één kop =
   firstobs=16. */
gegevens tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    lengte iso2 $5 jurisdiction_name $50;
    invoer iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
uitvoeren;

procedure afdrukken gegevens=tax_haven (obs=10);
    titel "Section 15 — jurisdiction risk weights (CTHI 2021)";
uitvoeren;

/* Kenmerken per entiteit met de naam van de topfunctionaris en het
   oprichtingsjaar. */
procedure gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Alles wat samen moet komen (pagerank, risicogewicht, vlag voor
   gesanctioneerde functionaris) gebeurt in één drievoudige LEFT JOIN
   met PROC SQL — geen DATA MERGE BY-keten, geen overbodige PROC SORTs,
   en COALESCE geeft ons de terugvalwaarden inline. */
procedure gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procedure sql;
    create table entity_flagged as
    selecteren e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case wanneer f.node_id is niet null dan 1 anders 0 einde
               as sanctioned_officer
    from   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Samengesteld risico: normaliseer de continue kenmerken min-max en
   weeg ze samen. PROC MEANS + één DATA-stap volstaat voor de
   normalisatie — dat is idiomatisch SAS. */
procedure gemiddelden gegevens=entity_flagged noprint;
    variabele top_officer_pr;
    uitvoer out=pr_max_ds max=pr_max;
uitvoeren;

gegevens entity_score;
    als _n_ = 1 dan instellen pr_max_ds;
    instellen entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    bewaren node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
uitvoeren;

/* Risicoverdeling over de volledige populatie van 24,957
   entiteiten. */
procedure gemiddelden gegevens=entity_score n min mean max;
    variabele risk officer_count risk_weight;
    titel "Section 15 — risk distribution across all entities";
uitvoeren;

/* De 25 entiteiten met het hoogste samengestelde risico. PROC SQL
   OUTOBS= vervangt een combinatie van PROC SORT + DATA-stap met
   obs=. */
procedure sql outobs=25;
    titel "Section 15 — top-25 composite-risk entities (names)";
    selecteren entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    order volgens risk desc;
quit;

/* Breng afzonderlijk de entiteiten naar voren die aan een
   gesanctioneerde functionaris zijn gekoppeld. */
procedure sql;
    titel "Section 15 — entities with at least one sanctioned officer";
    selecteren entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    waar  sanctioned_officer = 1
    order volgens risk desc;
quit;

## 16. Classificatie van doorvoer- versus eindjurisdicties

**Referentie:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo et al. verdelen de wereldwijde
bedrijfseigendomsgraph in "sink-OFC's" — waar bedrijfswaarde eindigt:
BM, KY, BVI, JE, IM — en "conduit-OFC's" — waardoorheen waarde stroomt:
NL, UK, CH, SG, IE. De Paradise-Papers-graph heeft een andere
populatie (voornamelijk bij Appleby gevestigde entiteiten), maar
hetzelfde structurele onderscheid zou de jurisdicties waar
functionarissen clusteren en adressen eindigen moeten scheiden van die
welke voornamelijk kapitaal doorsluizen.

Voor elke jurisdictie berekenen we vijf structurele kenmerken
rechtstreeks uit de live graph:

| Kenmerk | Signaal van |
|---|---|
| `log(n_entity)` | absolute omvang van de offshore-aanwezigheid van de jurisdictie |
| `avg_officers` | dichtheid van functionarissen per entiteit (sink-jurisdicties hebben meer functionarissen per entiteit) |
| `avg_xborder_off` | gemiddeld aantal functionarissen wier eigen land verschilt van de jurisdictie van de entiteit (conduit-indicator) |
| `intermediary_share` | aandeel entiteiten met een CONNECTED_TO-koppeling naar een tussenpersoon |
| `address_share` | aandeel entiteiten met een REGISTERED_ADDRESS-koppeling (sink-indicator) |

We standaardiseren naar z-scores en passen daarna Ward's
hiërarchische clustering met minimale variantie toe. `PROC TREE`
tekent het dendrogram. Merk op dat de Intermediary-knopen van de
Paradise Papers via `CONNECTED_TO` met Entity-knopen verbinden — niet
via `INTERMEDIARY_OF`, dat in het schema bestaat maar voor dit lek
geen gegevens draagt — dus gebruiken we hier `CONNECTED_TO`.

In [ ]:
/* Haal structurele kenmerken per jurisdictie uit de live graph. */
procedure gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Houd alleen jurisdicties met ten minste tien entiteiten aan zodat
   de gestandaardiseerde z-scores betekenisvol zijn. Het
   Paradise-Papers-lek telt in totaal 44 jurisdicties; 23 halen deze
   drempel. */
gegevens s16_filtered;
    instellen s16_raw;
    als n_entity >= 10;
    log_n_entity = log(n_entity);
uitvoeren;

procedure gemiddelden gegevens=s16_filtered noprint;
    variabele log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    uitvoer out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
uitvoeren;

gegevens s16_std;
    als _n_ = 1 dan instellen s16_stats;
    instellen s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    bewaren jurisdiction z1 z2 z3 z4 z5;
uitvoeren;

procedure afdrukken gegevens=s16_std;
    titel "Section 16 — standardised jurisdiction features";
uitvoeren;

/* Ward's hiërarchische clustering met minimale variantie. */
procedure cluster gegevens=s16_std method=ward outtree=s16_tree;
    variabele z1 z2 z3 z4 z5;
    id jurisdiction;
    titel "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
uitvoeren;

/* Teken het dendrogram. */
procedure tree gegevens=s16_tree horizontal;
    id jurisdiction;
    titel "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
uitvoeren;

## 17. Hoofdcomponenten van functionaris-netwerkrollen

**Referentie:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Zie ook Newman M E J, *Networks: An Introduction* (Oxford, 2010),
hoofdstuk 7.

Functionarissen in de Paradise-Papers-graph spelen structureel
verschillende rollen. Sommigen zitten in het centrum van een groot
cluster van verwante entiteiten; anderen verbinden anderszins
gescheiden clusters met elkaar (makelaars, in de zin van
Burt/Borgatti); een enkeling vormt de dichte kern van het
insider-netwerk van een bepaalde jurisdictie. Vier graph-metrieken
vangen deze verschillende rollen:

| Metriek | Vangt |
|---|---|
| `degree` | Aantal uitgaande `OFFICER_OF`-randen — breedte van affiliatie |
| `betweenness` | Fractie van kortste paden die door de functionaris lopen — **makelaardij** |
| `kcore` | Grootste k waarvoor de functionaris in een k-verbonden subgraph zit — **kern-inbedding** |
| `pagerank` | Eigenvector-achtige score uit dezelfde projectie — **invloed via invloedrijke partners** |

We berekenen alle vier de metrieken op de volledige ongerichte
projectie `(Officer)—[OFFICER_OF]—(Entity)` via de Neo4j Graph Data
Science-bibliotheek, beperken tot de 5,000 functionarissen met de
hoogste graad, en draaien `PROC PRINCOMP` op de log-getransformeerde
metrieken.

De PCA comprimeert de vier gecorreleerde metrieken tot orthogonale
assen wier relatieve ladingen ons vertellen welke rollen empirisch
samen clusteren en welke structureel verschillend zijn.

*Opmerking over de lokale clusteringscoëfficiënt:* Garcia-Bernardo et
al. nemen de lokale clusteringscoëfficiënt op als vijfde metriek. De
projectie `(Officer)—[:OFFICER_OF]—(Entity)` van de Paradise Papers is
strikt bipartiet, dus er kunnen geen driehoeken bestaan — elke lokale
clusteringscoëfficiënt is 0. We laten hem uit de metriekset weg.

In [ ]:
/* PROC NETWORK haalt een subgraph van de top-5000-functionarissen op
   via een alleen-lezen MATCH en berekent graad,
   eigenvector-centraliteit en k-core in-proces. Vervangt een eerdere
   gds.graph.project + vier gds.<algorithm>.stream-aanroepen — dat
   patroon vereist een GDS-projectiestap in schrijfmodus die de
   alleen-lezen step-neo4j-service van het platform weigert.

   Betweenness-centraliteit is opzettelijk weggelaten: NetworkX
   berekent exacte betweenness in O(V·E), wat de looptijd op deze
   subgraph domineert (5,000 functionarissen × ~78,000 randen). De PCA
   heeft nog steeds drie orthogonale assen — graad (lokale
   prominentie), eigenvector-centraliteit (globale invloed) en k-core
   (structurele cohesie) — genoeg om functionarisrollen te scheiden
   voor de kerninterpretatie. */
procedure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id tot=b_node_id;
    centrality degree eigen=unweight;
    core;
uitvoeren;

/* Haal de node-ids/namen van functionarissen op om te filteren. */
procedure gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filter op functionarisrijen. Eigenvector-centraliteit staat in de
   plaats van PageRank — zie de toelichting bij sectie 4. */
procedure sql;
    create table s17_metrics as
    selecteren n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order volgens n.centr_degree desc;
quit;

gegevens s17_metrics;
    instellen s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    bewaren node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
uitvoeren;

procedure afdrukken gegevens=s17_metrics (obs=10);
    titel "Section 17 — top-10 officers by degree (raw + log metrics)";
uitvoeren;

procedure gemiddelden gegevens=s17_metrics n mean std min max;
    variabele log_degree log_pr k_val;
    titel "Section 17 — log-transformed metric summary";
uitvoeren;

procedure princomp gegevens=s17_metrics out=s17_scores;
    variabele log_degree log_pr k_val;
    titel "Section 17 — PCA of officer network roles";
uitvoeren;

procedure sgplot gegevens=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis label="PC1 (global influence axis)";
    yaxis label="PC2 (brokerage vs core embeddedness)";
    titel "Section 17 — officers in 2D principal-component role space";
uitvoeren;

## 18. ARIMA-interventieanalyse op oprichtingssnelheden

**Referentie:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Toegepast op offshore-lekken door Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

Het jaarlijkse aantal nieuwe entiteiten in de Paradise-Papers-graph is
een bijna-monotone groeireeks van 1970 (36 entiteiten) tot 2007
(1,595 entiteiten, de piek), gevolgd door een daling in 2008-2009 en
een tragere afvlakking tot 2014 (het einde van de lekdekking).

We passen klassieke Box-Tiao-interventieanalyse toe om te toetsen of
twee gebeurtenissen uit de echte wereld meetbare sporen in de
oprichtingsreeks hebben achtergelaten:

- **Stap 2009** — de aanpak van belastingparadijzen tijdens de
  G20-top in Londen (april 2009), die samenviel met de financiële
  crisis.
- **Stap 2014** — de Amerikaanse FATCA (Foreign Account Tax Compliance
  Act) trad op 1 juli 2014 in werking.

De respons is `log(n)`, zodat een interventiecoëfficiënt van -0.30
overeenkomt met ruwweg een daling van 26 % in de jaarlijkse
oprichtingssnelheid. We passen `ARIMA(1,0,0)` toe — het
AR(1)-autoregressieve model op de sterk gecorreleerde reeks — met de
twee stap-dummy's als exogene `INPUT=`-variabelen.

De nulhypothese is dat geen van beide stappen een meetbare
verschuiving oplevert zodra de AR(1)-trend is verdisconteerd.
Gepubliceerde methodologieën verschillen over hoe een niet-verwerping
te interpreteren is: het kan betekenen dat de interventie geen effect
had, of dat de AR(1)-autocorrelatie het interventiesignaal absorbeert.

In [ ]:
procedure gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Bouw de dataset voor de interventiereeks. Twee stap-dummy's:
   step_2009 = I{year >= 2009} vangt de regimewijziging rond de
   aankondiging G20 Londen/FATCA; step_2014 = I{year >= 2014} vangt de
   schok van de ingangsdatum van FATCA. De respons is log(n), dus de
   coëfficiëntschattingen lezen als proportionele effecten. */
gegevens s18_ts;
    instellen year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
uitvoeren;

procedure afdrukken gegevens=s18_ts;
    titel "Section 18 — annual new-entity formations + intervention dummies";
uitvoeren;

procedure sgplot gegevens=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x label="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x label="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis label="Incorporation year";
    yaxis label="New entities per year";
    titel "Section 18 — Paradise-Papers annual formations, 1970-2014";
uitvoeren;

/* Identificeer het model en schat daarna ARIMA(1,0,0) met de twee
   stap-inputs. De CROSSCORR= van PROC ARIMA registreert de exogene
   variabelen bij de IDENTIFY-stap zodat ze beschikbaar zijn voor
   ESTIMATE INPUT=. */
procedure arima gegevens=s18_ts;
    identify variabele=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimate p=1 invoer=(step_2009 step_2014);
    titel "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
uitvoeren;
quit;

## 19. Zero-inflated telmodel voor sanctieblootstelling

**Referentie:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2e editie, Cambridge University Press (2013), hoofdstuk 4.
Zero-inflated modellen: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

Sectie 14 vond slechts **vijf** Paradise-Papers-entiteiten met ten
minste één functionaris op een geconsolideerde sanctielijst — een
verdwijnend zeldzame gebeurtenis. Een standaard Poisson- of
negatief-binomiale regressie op `sanctioned_count` per entiteit zou
slecht passen omdat **99.98 %** van de entiteiten nul heeft. Het
zero-inflated negatief-binomiale (ZINB) model gaat hiermee om door de
fit in twee delen te splitsen:

1. Een logistisch model voor de vraag of de entiteit tot een klasse
   van "structurele nullen" behoort (geen mogelijke
   sanctieblootstelling).
2. Een negatief-binomiaal model voor de telling onder de rest.

Met slechts 5 positieve gebeurtenissen over 21,538 entiteiten is de
ZINB-likelihood degeneratief — beide delen storten in. Die mislukking
is een **eerlijke eigenschap van de data**, niet van de procedure. We
draaien de ZINB-fit toch om te laten zien dat het regressiegereedschap
volledig werkt, en vallen daarna terug op een gewone binair-logistische
regressie op `has_sanctioned` (indicator voor `sanctioned_count > 0`).
De logistische regressie identificeert een helder effect: **elke extra
log-eenheid van het aantal functionarissen vermenigvuldigt de kans op
ten minste één gesanctioneerde functionaris met ongeveer 3.1**
(p = 0.002).

Covariaten:

- `top5` — klasse-variabele met 6 niveaus (BM, KY, VG, IM, JE, OTHER),
  referentiecategorie OTHER.
- `log_n_off` — `log(officer_count)`, de dominante omvangsvoorspeller.

In [ ]:
/* Haal per entiteit het aantal gesanctioneerde functionarissen uit
   de live graph. */
procedure gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

gegevens s19;
    instellen s19_raw;
    als officer_count >= 1;
    lengte top5 $5;
    top5 = 'OTHER';
    als jurisdiction = 'BM' dan top5 = 'BM';
    anders als jurisdiction = 'KY' dan top5 = 'KY';
    anders als jurisdiction = 'VG' dan top5 = 'VG';
    anders als jurisdiction = 'IM' dan top5 = 'IM';
    anders als jurisdiction = 'JE' dan top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
uitvoeren;

procedure frequenties gegevens=s19;
    tables top5 sanctioned_count has_sanctioned;
    titel "Section 19 — response-variable distribution (very zero-heavy)";
uitvoeren;

/* Eerst ZINB — naar verwachting degeneratief op een reeks met 5
   gebeurtenissen. */
procedure genmod gegevens=s19;
    klasse top5;
    model sanctioned_count = top5 log_n_off / dist=zinb link=log;
    titel "Section 19 — ZINB count model (degenerate on 5 events)";
uitvoeren;

/* Terugval: binair logistisch op has_sanctioned. Interpreteerbaar. */
procedure logistic gegevens=s19 aflopend plots=none;
    klasse top5;
    model has_sanctioned = top5 log_n_off;
    titel "Section 19 — logistic regression (has-sanctioned fallback)";
uitvoeren;

## 20. Poisson-oprichtingssnelheidsmodel met gemengde effecten

**Referentie:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Klassiek panel-telling: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

Sectie 18 paste een univariate ARIMA toe op de geaggregeerde
oprichtingsreeks. Hier gebruiken we de **panel**-dimensie: één rij per
jurisdictie-jaar-cel, waarbij we een gegeneraliseerd lineair gemengd
Poisson-model (GLMM) fitten met een vaste lineaire jaartrend plus een
FATCA-stap-dummy, en een **willekeurig intercept per jurisdictie**. Dit
scheidt het gemeenschappelijke-trend-effect van het niveau van de
individuele jurisdictie.

Panelbeperking: de 10 jurisdicties waarvan de **gemiddelde jaarlijkse
telling** >=5 is over 1990-2014 (in totaal 203 cellen). Kleinere
jurisdicties met veel jaren met nul-tellingen zouden een Poisson-fit
destabiliseren.

`PROC GLIMMIX` met `DIST=POISSON LINK=LOG` en
`RANDOM INTERCEPT / SUBJECT=jurisdiction` levert zowel de vaste
effecten op populatieniveau (jaartrend, FATCA-verschuiving) als de
variantiecomponent tussen jurisdicties. De variantie van het
willekeurige intercept vertelt ons *hoezeer oprichtingssnelheden
tussen jurisdicties verschillen na verdiscontering van de
gemeenschappelijke tijdtrend* — een score voor structurele
heterogeniteit voor de populatie van offshore financiële centra.

In [ ]:
/* Paneldataset: cellen jurisdictie x jaar van 1990-2014. */
procedure gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Houd jurisdicties met een gemiddelde jaarlijkse telling >= 5. */
procedure sql;
    create table s20_keep_jur as
    selecteren jurisdiction, avg(n) as avg_n
    from s20_raw
    group volgens jurisdiction
    having avg(n) >= 5;
quit;

procedure sql;
    create table s20 as
    selecteren r.*,
           r.year - 2002 as year_c,
           case wanneer r.year >= 2014 dan 1 anders 0 einde as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

procedure frequenties gegevens=s20;
    tables jurisdiction;
    titel "Section 20 — jurisdictions retained in the panel";
uitvoeren;

/* Poisson-GLMM met gemengde effecten: vaste jaartrend + FATCA-stap,
   willekeurig intercept per jurisdictie. */
procedure glimmix gegevens=s20;
    klasse jurisdiction;
    model n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    titel "Section 20 — mixed Poisson formation-rate model";
uitvoeren;

/* Gerangschikte willekeurige-intercept-effecten zouden de
   jurisdicties naar voren brengen die systematisch beter of slechter
   presteren dan de gemeenschappelijke trend. De SOLUTION-statement van
   PROC GLIMMIX drukt deze af in de standaarduitvoer hierboven — we
   laten de rangschikking aan de lezer over. */

In [ ]:
/* Sluit de rapport-PDF en geef de Neo4j-bibliotheek vrij. */
ods pdf close;

libname icij clear;

## Reproduceerbaarheid en herkomst

- **Graph-gegevensbron:** de ICIJ-onderzoeksdatabase *Offshore Leaks*,
  dataset *Paradise Papers*, voor het eerst uitgebracht in november
  2017. Beschikbaar op <https://offshoreleaks.icij.org/pages/database>.
  Geladen in de gedeelde `step-neo4j`-service van het platform
  (Neo4j 5.26 Community Edition, alleen-lezen op serverniveau) via de
  upstream publieke dump op
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **OFAC-SDN-gegevens:** publieke CSV-export van de Specially
  Designated Nationals van het Amerikaanse ministerie van Financiën
  (OFAC), opgehaald uit de publieke preview-API van het ministerie in
  april 2026. Het bestand `data/ofac_sdn.csv` bevat 500 gecureerde
  rijen over de vijf grootste OFAC-programma's naar aantal
  vermeldingen. Gebruikt voor de tweetraps-screening van sectie 6.
- **OpenSanctions-gegevens:** momentopname van de *default*
  geconsolideerde OpenSanctions-dataset van 2026-04-17, gedownload van
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Het vastgelegde bestand `data/opensanctions_default.csv` bevat de
  18,654 rijen met Person- en Company-schema waarvan de primaire
  dataset een van de sanctielijsten van OFAC SDN, het Britse HM
  Treasury, de EU-financiële sancties, de VN-Veiligheidsraad, of de
  geconsolideerde sanctielijsten van Canada, België, Australië,
  Zwitserland of andere landen is. Aliassen zijn weggelaten om het
  bestand onder de 2 MB te houden. Licentie: ODbL 1.0 (OpenSanctions).
  Gebruikt voor de verrijking van sectie 14.
- **Belastingparadijs-rangen:** de gepubliceerde ranglijsten van de
  *Corporate Tax Haven Index 2021* van het Tax Justice Network,
  samengesteld uit de indexlandingspagina `https://cthi.taxjustice.net`
  en het persbericht van maart 2021 op
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Het vastgelegde bestand `data/tax_haven_ranks.csv` bevat de
  jurisdicties die in de Paradise Papers voorkomen met hun CTHI-rang
  en een afgeleid risicogewicht in `[0, 1]`. Licentie: CC BY-SA
  4.0 (Tax Justice Network). Gebruikt voor de samengestelde ranglijst
  van sectie 15.
- **Graph-algoritmen:** Louvain-gemeenschapsdetectie en
  eigenvector-centraliteit (het dichtstbijzijnde interne equivalent van
  PageRank) berekend door `PROC NETWORK` in-proces op randlijsten die
  via alleen-lezen Cypher zijn opgehaald. De gedeelde
  `step-neo4j`-service van het platform is alleen-lezen op serverniveau,
  dus alle graph-algoritmen draaien in de workspace-pod in plaats van
  via schrijfbewerkingen van Neo4j Graph Data Science.
- **Statistische methoden:** `PROC LIFETEST` gebruikt de
  Kaplan-Meier-schatter met gestratificeerde log-rank-, Wilcoxon- en
  Tarone-Ware-toetsen. `PROC PHREG` fit het Cox proportional-hazards-
  model via Breslow-ties met de Python/statsmodels-wrapper.
  `PROC LOGISTIC` fit een binair-logistische regressie via
  maximale-likelihood.
- **Methoden voor risicosamenstelling:** de samengestelde
  invloedsscore van sectie 11 normaliseert graad, log-PageRank,
  jurisdictiebreedte en duur in jaren naar `[0, 1]` en telt op met
  vaste gewichten (30/30/20/20). De samengestelde risicoscore per
  entiteit van sectie 15 normaliseert het afgetopte aantal
  functionarissen, log-PageRank, het CTHI-risicogewicht en een binaire
  vlag voor gesanctioneerde functionarissen, en telt op met gelijke
  gewichten van elk 0.25.

De volledige analyse is reproduceerbaar met het smoke-test-script in
deze map: `./smoke.jenner`. Het end-to-end draaien tegen de gedeelde
`step-neo4j`-service (met `JENNER_NEO4J_HOST` en `JENNER_NEO4J_PASS`
ingesteld, wat het platform voor je doet in elke workspace-pod) duurt
ruwweg twee minuten en verifieert dat elke query en elke PROC —
inclusief de vijf nieuwe secties die naast de bestaande pijplijn zijn
toegevoegd — de verwachte uitvoer op de echte graph van 163,435 knopen
oplevert.

